# Formula 1: five raw files into one master table
Loads results, races, drivers, constructors and circuits (CSV, tab-separated text and JSON), replaces `\N` markers, drops unused columns, fixes types, and merges everything into a master table.

**Data:** not included, see the repository README for sources.

In [1]:
%cd /content/data

/content/data


In [2]:
import pandas as pd
import numpy as np

# --- Load the datasets ---

# results.csv is a standard CSV
df_results = pd.read_csv('results.csv')

# races.csv is a standard CSV
df_races = pd.read_csv('races.csv')

# constructors.csv is a standard CSV
df_constructors = pd.read_csv('constructors.csv')

# circuits.json is a standard JSON file
df_circuits = pd.read_json('circuits.json')

# drivers.txt is a tab-separated file. We must specify the separator.
# We also replace the '\\N' string with a proper NaN (Not a Number) value during import.
df_drivers = pd.read_csv('d.csv')


# --- Initial Data Inspection ---
print("--- Initial DataFrames ---")
print("Results Shape:", df_results.shape)
print("Races Shape:", df_races.shape)
print("Constructors Shape:", df_constructors.shape)
print("Drivers Shape:", df_drivers.shape)
print("Circuits Shape:", df_circuits.shape)

--- Initial DataFrames ---
Results Shape: (25420, 18)
Races Shape: (1079, 18)
Constructors Shape: (211, 5)
Drivers Shape: (854, 9)
Circuits Shape: (79, 9)


In [6]:
# Replace '\\N' with NaN (Not a Number) for columns that weren't caught during import
df_results.replace('\\N', np.nan, inplace=True)

# Drop irrelevant columns
columns_to_drop_results = ['number', 'grid', 'position', 'positionText', 'laps', 'time',
                           'milliseconds', 'fastestLap', 'rank', 'fastestLapTime',
                           'fastestLapSpeed', 'statusId']

# Filter columns to drop to include only those present in the DataFrame
columns_to_drop_results_present = [col for col in columns_to_drop_results if col in df_results.columns]
df_results.drop(columns=columns_to_drop_results_present, inplace=True)


# Create 'Win' and 'Podium' columns
df_results['Win'] = np.where(df_results['positionOrder'] == 1, 1, 0)
df_results['Podium'] = np.where(df_results['positionOrder'] <= 3, 1, 0)

# Correct data types
df_results['points'] = df_results['points'].astype(float)
df_results['positionOrder'] = df_results['positionOrder'].fillna(0).astype(int)
df_results[['resultId', 'raceId', 'driverId', 'constructorId']] = df_results[['resultId', 'raceId', 'driverId', 'constructorId']].astype(str)
df_results[['Win', 'Podium']] = df_results[['Win', 'Podium']].astype(int)


print("\n--- Cleaned Results DataFrame ---")
print(df_results.info())
print(df_results.head())


--- Cleaned Results DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25420 entries, 0 to 25419
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   resultId       25420 non-null  object 
 1   raceId         25420 non-null  object 
 2   driverId       25420 non-null  object 
 3   constructorId  25420 non-null  object 
 4   positionOrder  25420 non-null  int64  
 5   points         24960 non-null  float64
 6   Win            25420 non-null  int64  
 7   Podium         25420 non-null  int64  
dtypes: float64(1), int64(3), object(4)
memory usage: 1.6+ MB
None
  resultId raceId driverId constructorId  positionOrder  points  Win  Podium
0   7554.0    1.0     18.0          23.0              1    10.0    1       1
1   7555.0    1.0     22.0          23.0              2     8.0    0       1
2   7556.0    1.0     15.0           7.0              3     6.0    0       1
3   7557.0    1.0     10.0           7.0  

In [7]:
# Drop irrelevant columns (including the unnamed empty ones at the end)
df_races.drop(columns=['time', 'url'], inplace=True)
df_races.dropna(axis=1, how='all', inplace=True) # Drops any column where all values are null

# Correct data types
df_races['date'] = pd.to_datetime(df_races['date'])
df_races[['raceId', 'circuitId']] = df_races[['raceId', 'circuitId']].astype(str)

# Clean text columns
df_races['name'] = df_races['name'].str.strip()


print("\n--- Cleaned Races DataFrame ---")
print(df_races.info())
print(df_races.head())


--- Cleaned Races DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1079 entries, 0 to 1078
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   raceId     1079 non-null   object        
 1   year       1035 non-null   float64       
 2   round      1035 non-null   float64       
 3   circuitId  1079 non-null   object        
 4   name       1035 non-null   object        
 5   date       1035 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(2), object(3)
memory usage: 50.7+ KB
None
  raceId    year  round circuitId                   name       date
0    1.0  2009.0    1.0       1.0  Australian Grand Prix 2009-03-29
1    2.0  2009.0    2.0       2.0   Malaysian Grand Prix 2009-04-05
2    3.0  2009.0    3.0      17.0     Chinese Grand Prix 2009-04-19
3    4.0  2009.0    4.0       3.0     Bahrain Grand Prix 2009-04-26
4    5.0  2009.0    5.0       4.0     Spanish Grand Prix 2009-05-10


/tmp/ipython-input-898894272.py:6: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_races['date'] = pd.to_datetime(df_races['date'])


In [8]:
# Drop irrelevant columns
df_constructors.drop(columns=['constructorRef', 'url'], inplace=True)

# Correct data types
df_constructors['constructorId'] = df_constructors['constructorId'].astype(str)

# Clean text columns
df_constructors['name'] = df_constructors['name'].str.strip()
df_constructors['nationality'] = df_constructors['nationality'].str.strip()

print("\n--- Cleaned Constructors DataFrame ---")
print(df_constructors.info())
print(df_constructors.head())


--- Cleaned Constructors DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   constructorId  211 non-null    object
 1   name           211 non-null    object
 2   nationality    211 non-null    object
dtypes: object(3)
memory usage: 5.1+ KB
None
  constructorId        name nationality
0             1     McLaren     British
1             2  BMW Sauber      German
2             3    Williams     British
3             4     Renault      French
4             5  Toro Rosso     Italian


In [25]:
# Drop irrelevant columns
columns_to_drop_drivers = ['driverRef', 'number', 'code', 'url']
columns_to_drop_drivers_present = [col for col in columns_to_drop_drivers if col in df_drivers.columns]
df_drivers.drop(columns=columns_to_drop_drivers_present, inplace=True)


# Create 'FullName' column (keeping forename and surname for merge)
df_drivers['FullName'] = df_drivers['forename'] + ' ' + df_drivers['surname']


# Correct data types
df_drivers['dob'] = pd.to_datetime(df_drivers['dob'], format='%d/%m/%Y', errors='coerce')
df_drivers['driverId'] = df_drivers['driverId'].astype(str)

# Clean text columns
df_drivers['FullName'] = df_drivers['FullName'].str.strip()
df_drivers['nationality'] = df_drivers['nationality'].str.strip()

print("\n--- Cleaned Drivers DataFrame ---")
print(df_drivers.info())
print(df_drivers.head())


--- Cleaned Drivers DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 854 entries, 0 to 853
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   driverId     854 non-null    object        
 1   forename     854 non-null    object        
 2   surname      854 non-null    object        
 3   dob          847 non-null    datetime64[ns]
 4   nationality  854 non-null    object        
 5   FullName     854 non-null    object        
dtypes: datetime64[ns](1), object(5)
memory usage: 40.2+ KB
None
  driverId  forename     surname        dob nationality           FullName
0        1     Lewis    Hamilton 1985-01-07     British     Lewis Hamilton
1        2      Nick    Heidfeld 1977-05-10      German      Nick Heidfeld
2        3      Nico     Rosberg 1985-06-27      German       Nico Rosberg
3        4  Fernando      Alonso 1981-07-29     Spanish    Fernando Alonso
4        5    Heikki  Kovalai

In [14]:
# Correct data types
df_circuits['circuitId'] = df_circuits['circuitId'].astype(str)
df_circuits[['lat', 'lng']] = df_circuits[['lat', 'lng']].astype(float)

# Clean text columns
df_circuits['name'] = df_circuits['name'].str.strip()
df_circuits['location'] = df_circuits['location'].str.strip()
df_circuits['country'] = df_circuits['country'].str.strip()

print("\n--- Cleaned Circuits DataFrame ---")
print(df_circuits.info())
print(df_circuits.head())


--- Cleaned Circuits DataFrame ---
<class 'pandas.core.frame.DataFrame'>
Index: 79 entries, 0 to 78
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   circuitId   79 non-null     object 
 1   circuitRef  79 non-null     object 
 2   name        79 non-null     object 
 3   location    79 non-null     object 
 4   country     79 non-null     object 
 5   lat         79 non-null     float64
 6   lng         79 non-null     float64
 7   alt         79 non-null     object 
 8   url         79 non-null     object 
dtypes: float64(2), object(7)
memory usage: 6.2+ KB
None
  circuitId   circuitRef                            name      location  \
0         1  albert_park  Albert Park Grand Prix Circuit     Melbourne   
1         2       sepang    Sepang International Circuit  Kuala Lumpur   
2         3      bahrain   Bahrain International Circuit        Sakhir   
3         4    catalunya  Circuit de Barcelona-Catalunya      M

In [34]:
# Merge results with races
df_master = pd.merge(df_results, df_races, on='raceId', how='left')

# Merge with drivers
df_master = pd.merge(df_master, df_drivers, on='driverId', how='left')

# Merge with constructors
df_master = pd.merge(df_master, df_constructors, on='constructorId', how='left', suffixes=('_race', '_constructor'))

# Merge with circuits
df_master = pd.merge(df_master, df_circuits, on='circuitId', how='left', suffixes=('_race', '_circuit'))

# Rename columns for clarity after merges
df_master.rename(columns={'name_race': 'race_name',
                          'name_constructor': 'constructor_name',
                          'name_circuit': 'circuit_name',
                          'nationality_constructor': 'constructor_nationality',
                          'nationality': 'driver_nationality'}, inplace=True)

print("\n--- Master DataFrame ---")
print(df_master.info())
print(df_master.head())


--- Master DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24960 entries, 0 to 24959
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   resultId                 24960 non-null  int64         
 1   raceId                   24960 non-null  object        
 2   driverId                 24960 non-null  object        
 3   constructorId            24960 non-null  object        
 4   positionOrder            24960 non-null  float64       
 5   points                   24960 non-null  float64       
 6   Win                      24960 non-null  int64         
 7   Podium                   24960 non-null  int64         
 8   year                     24960 non-null  float64       
 9   round                    24960 non-null  float64       
 10  circuitId                24960 non-null  object        
 11  race_name                24960 non-null  object        
 12  date  

In [35]:
# Ensure IDs match by converting to string
df_results[['raceId','driverId','constructorId']] = df_results[['raceId','driverId','constructorId']].astype(str)
df_races['raceId'] = df_races['raceId'].astype(str)
df_races['circuitId'] = df_races['circuitId'].astype(str)
df_drivers['driverId'] = df_drivers['driverId'].astype(str)
df_constructors['constructorId'] = df_constructors['constructorId'].astype(str)
df_circuits['circuitId'] = df_circuits['circuitId'].astype(str)

# Do the merges again fresh
df_master = df_results.merge(df_races, on='raceId', how='left') \
                      .merge(df_drivers, on='driverId', how='left') \
                      .merge(df_constructors, on='constructorId', how='left') \
                      .merge(df_circuits, on='circuitId', how='left')

# --- Inspection after merge ---
print("\n--- df_master info after merge ---")
print(df_master.info())
print("\n--- df_master head after merge ---")
print(df_master.head())

# --- Summary Analysis ---
total_races = df_master['raceId'].nunique()
total_drivers = df_master['driverId'].nunique()
total_constructors = df_master['constructorId'].nunique()
print(f"Total Races: {total_races}")
print(f"Total Drivers: {total_drivers}")
print(f"Total Constructors: {total_constructors}")


# --- Driver Analysis ---
print("\n--- Top 10 Drivers by Wins ---")
# Use forename and surname from merged dataframe for FullName
df_master['DriverFullName'] = df_master['forename'] + ' ' + df_master['surname']
driver_wins = df_master.groupby('DriverFullName')['Win'].sum().sort_values(ascending=False)
print(driver_wins.head(10))

print("\n--- Top 10 Drivers by Podiums ---")
driver_podiums = df_master.groupby('DriverFullName')['Podium'].sum().sort_values(ascending=False)
print(driver_podiums.head(10))


# --- Constructor Analysis ---
print("\n--- Top 10 Constructors by Total Points ---")
constructor_points = df_master.groupby('name_y')['points'].sum().sort_values(ascending=False)
print(constructor_points.head(10))


--- df_master info after merge ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24960 entries, 0 to 24959
Data columns (total 30 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   resultId        24960 non-null  int64         
 1   raceId          24960 non-null  object        
 2   driverId        24960 non-null  object        
 3   constructorId   24960 non-null  object        
 4   positionOrder   24960 non-null  float64       
 5   points          24960 non-null  float64       
 6   Win             24960 non-null  int64         
 7   Podium          24960 non-null  int64         
 8   year            24960 non-null  float64       
 9   round           24960 non-null  float64       
 10  circuitId       24960 non-null  object        
 11  name_x          24960 non-null  object        
 12  date            24960 non-null  datetime64[ns]
 13  forename        24960 non-null  object        
 14  surname         24

In [36]:
# Ferrari Wins
ferrari_wins = df_master[df_master['name_y'] == 'Ferrari']['Win'].sum()
print("Ferrari Wins:", ferrari_wins)

# Races hosted by Monza
monza_races = df_master[df_master['name'] == 'Autodromo Nazionale Monza']['raceId'].nunique()
print("Monza Races Hosted:", monza_races)


Ferrari Wins: 239
Monza Races Hosted: 0


In [37]:
# --- Driver Analysis: Top 10 by Wins ---
print("\n--- Top 10 Drivers by Wins ---")
driver_wins = df_master.groupby('DriverFullName')['Win'].sum().sort_values(ascending=False)
print(driver_wins.head(10))

# --- Constructor Analysis: Top 10 by Wins ---
print("\n--- Top 10 Constructors by Wins ---")
constructor_wins = df_master.groupby('name_y')['Win'].sum().sort_values(ascending=False)
print(constructor_wins.head(10))

# --- Circuit Analysis: Which circuits hosted the most races? ---
print("\n--- Top 10 Circuits by Number of Races Hosted ---")
circuit_races = df_master['name'].value_counts()
print(circuit_races.head(10))

# --- Validation Checks ---
print("\n--- Validation Checks ---")
hamilton_wins = driver_wins.get('Lewis Hamilton', 0)
ferrari_wins = constructor_wins.get('Ferrari', 0)
monza_races = circuit_races.get('Autodromo Nazionale di Monza', 0)

print(f"Lewis Hamilton Total Wins: {hamilton_wins}")
print(f"Ferrari Total Wins: {ferrari_wins}")
print(f"Races at Monza: {monza_races}")


--- Top 10 Drivers by Wins ---
DriverFullName
Lewis Hamilton        95
Michael Schumacher    91
Sebastian Vettel      53
Alain Prost           51
Ayrton Senna          41
Fernando Alonso       32
Nigel Mansell         31
Jackie Stewart        27
Niki Lauda            25
Jim Clark             25
Name: Win, dtype: int64

--- Top 10 Constructors by Wins ---
name_y
Ferrari       239
McLaren       178
Mercedes      115
Williams      114
Red Bull       64
Team Lotus     45
Renault        35
Benetton       27
Brabham        23
Tyrrell        23
Name: Win, dtype: int64

--- Top 10 Circuits by Number of Races Hosted ---
name
Autodromo Nazionale di Monza    1756
Circuit de Monaco               1584
Silverstone Circuit             1356
Circuit de Spa-Francorchamps    1178
Circuit Gilles Villeneuve        992
Nürburgring                      976
Hockenheimring                   935
Autódromo José Carlos Pace       857
Hungaroring                      831
Suzuka Circuit                   731
Name:

In [38]:
# Monza ke liye sirf unique raceId gino
monza_races = df_master[df_master['circuitRef'] == 'monza']['raceId'].nunique()
print("Unique Monza Races Hosted:", monza_races)


Unique Monza Races Hosted: 70


In [41]:
# Ensure IDs match by converting to string (using the successful approach from the previous cell)
df_results[['raceId','driverId','constructorId']] = df_results[['raceId','driverId','constructorId']].astype(str)
df_races['raceId'] = df_races['raceId'].astype(str)
df_races['circuitId'] = df_races['circuitId'].astype(str)
df_drivers['driverId'] = df_drivers['driverId'].astype(str)
df_constructors['constructorId'] = df_constructors['constructorId'].astype(str)
df_circuits['circuitId'] = df_circuits['circuitId'].astype(str)

# Re-merge the dataframes to ensure consistency after type conversions
df_master = df_results.merge(df_races, on='raceId', how='left') \
                      .merge(df_drivers, on='driverId', how='left') \
                      .merge(df_constructors, on='constructorId', how='left') \
                      .merge(df_circuits, on='circuitId', how='left')

# Create 'DriverFullName' column in the merged dataframe for consistency
df_master['DriverFullName'] = df_master['forename'] + ' ' + df_master['surname']

# Rename confusing columns
df_master = df_master.rename(columns={
    'name_x': 'race_name',
    'name_y': 'constructor_name',
    'name': 'circuit_name',
    'nationality_x': 'driver_nationality',
    'nationality_y': 'constructor_nationality',
    'url_x': 'race_url',
    'url_y': 'circuit_url'
})

# alt ko numeric (errors ignore)
df_master['alt'] = pd.to_numeric(df_master['alt'], errors='coerce')


# ✅ Step 4: Validation checks
ham_wins = df_master[df_master['DriverFullName']=="Lewis Hamilton"]['Win'].sum()
ferrari_wins = df_master[df_master['constructor_name']=="Ferrari"]['Win'].sum()
monza_races = df_master[df_master['circuit_name'].str.contains("Monza")]['raceId'].nunique()

print("Lewis Hamilton Wins:", ham_wins)
print("Ferrari Wins:", ferrari_wins)
print("Monza Races Hosted:", monza_races)



# ✅ Step 5: Export to CSV
df_master.to_csv("master_f1_cleaned.csv", index=False)
print("Exported: master_f1_cleaned.csv")

Lewis Hamilton Wins: 95
Ferrari Wins: 239
Monza Races Hosted: 70
Exported: master_f1_cleaned.csv
